# Pipeline CPU 2 : PyTorch -> torch.compile (Inductor) -> CPU
Runtime : CPU. Variante TorchScript en bonus en fin de notebook.

In [1]:
from google.colab import drive
drive.mount('/content/drive')
import sys, os
sys.path.append('/content/drive/MyDrive/tp_deploy')
from common import *
X, y = load_data()
print(X.shape, y[:10])

Mounted at /content/drive
torch.Size([50, 3, 224, 224]) [2 1 5 8 9 3 8 9 0 1]


In [2]:
model = load_model()
cmodel = torch.compile(model, backend='inductor')
t = time.perf_counter()
with torch.inference_mode():
    cmodel(X[:BS])
print('première inférence (compilation incluse) :', time.perf_counter() - t, 's')

def predict(b):
    with torch.inference_mode():
        return cmodel(b).numpy()

première inférence (compilation incluse) : 33.17438674499999 s


In [3]:
res = run_benchmark('cpu_torchcompile', predict, X, y, device='cpu')

RAPL indisponible : [Errno 2] No such file or directory: '/sys/class/powercap/intel-rapl:0/energy_uj'
total_s      moy=3.5781 std=0.0587 min=3.5263 max=3.6468
throughput   moy=13.9771 std=0.2279 min=13.7106 max=14.1792
batch_ms     moy=715.5980 std=11.7335 min=705.2474 max=729.3528
top1         moy=0.9200 std=0.0000 min=0.9200 max=0.9200
énergie : None


## Bonus : variante TorchScript

In [ ]:
ts = torch.jit.freeze(torch.jit.trace(load_model(), torch.randn(BS,3,224,224)))
def predict_ts(b):
    with torch.inference_mode():
        return ts(b).numpy()
res_ts = run_benchmark('cpu_torchscript', predict_ts, X, y, device='cpu')

RAPL indisponible : [Errno 2] No such file or directory: '/sys/class/powercap/intel-rapl:0/energy_uj'
total_s      moy=4.4205 std=0.1911 min=4.2192 max=4.6130
throughput   moy=11.3280 std=0.4929 min=10.8388 max=11.8507
batch_ms     moy=884.0841 std=38.2177 min=843.8188 max=922.5934
top1         moy=0.9200 std=0.0000 min=0.9200 max=0.9200
énergie : None
